# Enhanced Green-Apple Results, Status, Videos, Depth, and ByteTrack

This notebook is the **results companion** for `AAM, EBCA, FM, KWConv 100.ipynb`.
It contains no project `.py` dependency. It reads the active enhanced-YOLOv8 training run, then—only after training finishes—creates the final validation graphs, CSV tables, and three requested annotated videos.

The primary notebook remains the training notebook. This separate notebook can be opened now to monitor training without interrupting the active GPU run.

## Run order and exact result locations

1. Run **Live training status** at any time. It reads the latest completed epoch from `Results/training/aam_ebca_fm_kwconv_green/results.csv` and writes `Results/status/live_training_status.json`.
2. Wait until the primary notebook has completed all configured epochs and `Results/training/aam_ebca_fm_kwconv_green/weights/best.pt` exists.
3. Run **Final validation and graphs** with `RUN_FINAL_VALIDATION = True`. It writes the required F1, precision, recall, PR, confusion-matrix, and normalized-confusion-matrix images into `Results/graphs/`.
4. Run **Three annotated videos and CSV tables** with `RUN_VIDEO_EXPORTS = True`. It runs enhanced detection, AdaBins relative depth, and depth-aware ByteTrack once over the original MP4 and writes all three videos and both CSV tables to `Results/inference/` and `Results/csv/`.

Do not run the validation or video-export cells while the 300-epoch training run is using the 4-GB GPU. They deliberately default to `False`.

| Requested artifact | Exact saved path |
|---|---|
| Live epoch status | `Results/status/live_training_status.json` |
| Training values by epoch | `Results/training/aam_ebca_fm_kwconv_green/results.csv` and `Results/csv/training_epoch_metrics.csv` |
| F1 / confidence | `Results/graphs/F1_curve.png` |
| Precision / confidence | `Results/graphs/P_curve.png` |
| Recall / confidence | `Results/graphs/R_curve.png` |
| Precision / recall | `Results/graphs/PR_curve.png` |
| Confusion matrix | `Results/graphs/confusion_matrix.png` |
| Normalized confusion matrix | `Results/graphs/confusion_matrix_normalized.png` |
| Raw apple bounding boxes, confidence and depth | `Results/csv/apple_detections_confidence_depth.csv` |
| ByteTrack IDs, boxes, confidence and depth | `Results/csv/apple_tracks_id_confidence_depth.csv` |
| Video: ID + depth | `Results/inference/apples_id_depth.mp4` |
| Video: confidence + depth | `Results/inference/apples_confidence_depth.mp4` |
| Video: ID + confidence | `Results/inference/apples_id_confidence.mp4` |

## Live training status

This is safe to run during training. Each CSV row represents a completed epoch; an epoch that is currently running is not written until it finishes. The displayed precision/recall/mAP values are validation values against automatically generated labels, not manually verified ground truth.

In [ ]:
from pathlib import Path
import csv
import json
from datetime import datetime, timezone

def locate_project(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'Codeing').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('Open this notebook from the project directory or a subdirectory.')

PROJECT_ROOT = locate_project(Path.cwd().resolve())
RUN_NAME = 'aam_ebca_fm_kwconv_green'
RUN_DIR = PROJECT_ROOT / 'Results' / 'training' / RUN_NAME
RESULTS_CSV = RUN_DIR / 'results.csv'
CONFIGURED_EPOCHS = 300
STATUS_DIR = PROJECT_ROOT / 'Results' / 'status'
STATUS_DIR.mkdir(parents=True, exist_ok=True)

rows = []
if RESULTS_CSV.exists():
    with RESULTS_CSV.open(encoding='utf-8', newline='') as handle:
        rows = list(csv.DictReader(handle))
latest = {key.strip(): value for key, value in rows[-1].items()} if rows else {}

def number(key: str) -> float | None:
    value = latest.get(key)
    return float(value) if value not in (None, '') else None

metric_columns = {
    'precision': 'metrics/precision(B)',
    'recall': 'metrics/recall(B)',
    'map50': 'metrics/mAP50(B)',
    'map50_95': 'metrics/mAP50-95(B)',
}
best_observed = {}
for name, column in metric_columns.items():
    candidates = [row for row in rows if row.get(column) not in (None, '')]
    if candidates:
        best = max(candidates, key=lambda row: float(row[column]))
        best_observed[name] = {'value': float(best[column]), 'epoch': int(float(best['epoch']))}

checkpoint_dir = RUN_DIR / 'weights'
latest_report = None
if latest:
    latest_report = {
        'epoch': int(float(latest['epoch'])),
        'elapsed_seconds': number('time'),
        'precision': number('metrics/precision(B)'),
        'recall': number('metrics/recall(B)'),
        'map50': number('metrics/mAP50(B)'),
        'map50_95': number('metrics/mAP50-95(B)'),
        'train_box_loss': number('train/box_loss'),
        'train_cls_loss': number('train/cls_loss'),
        'train_dfl_loss': number('train/dfl_loss'),
        'val_box_loss': number('val/box_loss'),
        'val_cls_loss': number('val/cls_loss'),
        'val_dfl_loss': number('val/dfl_loss'),
    }
report = {
    'schema_version': 1,
    'checked_utc': datetime.now(timezone.utc).isoformat(),
    'training_csv': str(RESULTS_CSV.relative_to(PROJECT_ROOT)),
    'configured_epochs': CONFIGURED_EPOCHS,
    'completed_epochs': len(rows),
    'progress_percent': round(100 * len(rows) / CONFIGURED_EPOCHS, 2),
    'training_complete': len(rows) >= CONFIGURED_EPOCHS,
    'latest': latest_report,
    'best_observed': best_observed,
    'checkpoints': {
        'best_pt_present_locally': (checkpoint_dir / 'best.pt').exists(),
        'last_pt_present_locally': (checkpoint_dir / 'last.pt').exists(),
        'published_to_git': False,
    },
    'evaluation_notice': 'Provisional validation metrics against automatically generated pseudo-labels; not independent ground truth.',
}
(STATUS_DIR / 'live_training_status.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))
if not rows:
    print(f'No completed epoch is written yet: {RESULTS_CSV.relative_to(PROJECT_ROOT)}')

## Notebook-only compatibility definitions for the enhanced checkpoint

The training checkpoint includes the AAM, EBCA, FM, and KWConv-enhanced Detect head. The next two code cells recreate those exact notebook definitions before `best.pt` is loaded. This is required for correct inference; do not skip them.

In [ ]:
# %% 02A) Core paths, manifests, and video metadata
"""Reusable helpers for the green-apple detection, depth, tracking, and count workflow.

All generated data is kept below ``Dataset/`` and all reports/artifacts below
``Results/``. The source MP4 is read-only.  The notebook is intentionally the
entry point; this module keeps each notebook cell short and reproducible.
"""

from __future__ import annotations

import csv
import platform
import json
import math
import os
import random
import shutil
import subprocess
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable, Sequence

import cv2
import numpy as np


CLASS_NAMES = ("green_apple",)
GREEN_APPLE = 0
DEFAULT_VARIANTS = (
    "hflip",
    "vflip",
    "bright",
    "dark",
    "rot90",
    "rot180",
    "rot270",
    "small_rotate",
    "motion_blur",
    "gaussian_noise",
)


@dataclass(frozen=True)
class ProjectPaths:
    """Canonical paths used by the rebuilt single-green-apple workflow."""

    root: Path

    @property
    def dataset(self) -> Path:
        return self.root / "Dataset"

    @property
    def results(self) -> Path:
        return self.root / "Results"

    @property
    def raw_video(self) -> Path:
        return self.dataset / "DJI_0418.MP4"

    @property
    def frames_80fps(self) -> Path:
        return self.dataset / "frames" / "80fps_timeline"

    @property
    def picked_800(self) -> Path:
        return self.dataset / "frames" / "picked_800_uniform"

    @property
    def augmented_800(self) -> Path:
        return self.dataset / "augmentations" / "from_800"

    @property
    def manifests(self) -> Path:
        return self.dataset / "manifests"

    @property
    def yolo(self) -> Path:
        return self.dataset / "yolo_green_apple"

    @property
    def yolo_images(self) -> Path:
        return self.yolo / "images"

    @property
    def yolo_labels(self) -> Path:
        return self.yolo / "labels"

    @property
    def auto_previews(self) -> Path:
        return self.yolo / "auto_label_previews"

    @property
    def data_yaml(self) -> Path:
        return self.yolo / "green_apple.yaml"

    @property
    def hardware_results(self) -> Path:
        return self.results / "hardware"

    @property
    def dataset_results(self) -> Path:
        return self.results / "dataset"

    @property
    def architecture_results(self) -> Path:
        return self.results / "architecture"

    @property
    def training_results(self) -> Path:
        return self.results / "training"

    @property
    def validation_results(self) -> Path:
        return self.results / "validation"

    @property
    def inference_results(self) -> Path:
        return self.results / "inference"

    def ensure_layout(self) -> None:
        folders = (
            self.dataset,
            self.results,
            self.frames_80fps,
            self.picked_800,
            self.augmented_800,
            self.manifests,
            self.auto_previews,
            self.hardware_results,
            self.dataset_results,
            self.architecture_results,
            self.training_results,
            self.validation_results,
            self.inference_results,
        )
        for folder in folders:
            folder.mkdir(parents=True, exist_ok=True)
        for split in ("train", "val", "test"):
            (self.yolo_images / split).mkdir(parents=True, exist_ok=True)
            (self.yolo_labels / split).mkdir(parents=True, exist_ok=True)


def find_project_root(start: str | Path | None = None) -> Path:
    """Find the project root when a notebook is launched from ``Codeing/``."""
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "Dataset").exists() and (candidate / "Codeing").exists():
            return candidate
    raise FileNotFoundError("Could not find a project root containing Dataset/ and Codeing/.")


def project_paths(start: str | Path | None = None) -> ProjectPaths:
    paths = ProjectPaths(find_project_root(start))
    paths.ensure_layout()
    return paths


def _write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2), encoding="utf-8")


def _write_jsonl(path: Path, rows: Iterable[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row) + "\n")


def _clear_generated_images(folder: Path, pattern: str = "*.jpg") -> None:
    """Delete only generated files in one known pipeline folder."""
    for item in folder.glob(pattern):
        item.unlink()


In [ ]:
# %% 02E) Custom AAM, EBCA, FM, KWConv modules and enhanced Detect head
def _require_torch() -> tuple[Any, Any, Any]:
    try:
        import torch
        from torch import nn
        from torch.nn import functional as functional
    except ModuleNotFoundError as error:
        raise RuntimeError("Install PyTorch before constructing the custom feature modules.") from error
    return torch, nn, functional


try:
    torch, nn, F = _require_torch()
except RuntimeError:
    torch = nn = F = None


if nn is not None:

    class AAM(nn.Module):
        """Attribute attention: colour, shape, and scale branches with learned fusion."""

        def __init__(self, channels: int, reduction: int = 8) -> None:
            super().__init__()
            hidden = max(1, channels // reduction)
            self.colour = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, hidden, 1), nn.SiLU(), nn.Conv2d(hidden, channels, 1))
            self.shape = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
            self.scale = nn.Sequential(nn.AvgPool2d(3, stride=1, padding=1), nn.Conv2d(channels, channels, 1))
            self.weights = nn.Parameter(torch.zeros(3))

        def forward(self, features: Any) -> Any:
            colour = self.colour(features).expand_as(features)
            shape = self.shape(features)
            scale = self.scale(features)
            alpha = torch.softmax(self.weights, dim=0)
            return features * torch.sigmoid(alpha[0] * colour + alpha[1] * shape + alpha[2] * scale)


    class EBCA(nn.Module):
        """Efficient bidirectional cross-attention between P3 and P4."""

        def __init__(self, p3_channels: int, p4_channels: int) -> None:
            super().__init__()
            self.p4_to_p3 = nn.Conv2d(p4_channels, p3_channels, 1, bias=False)
            self.p3_to_p4 = nn.Conv2d(p3_channels, p4_channels, 1, bias=False)
            self.p3_query = nn.Conv2d(p3_channels, p3_channels, 1, bias=False)
            self.p4_query = nn.Conv2d(p4_channels, p4_channels, 1, bias=False)
            self.p3_norm = nn.BatchNorm2d(p3_channels)
            self.p4_norm = nn.BatchNorm2d(p4_channels)

        def forward(self, p3: Any, p4: Any) -> tuple[Any, Any]:
            p4_up = F.interpolate(self.p4_to_p3(p4), size=p3.shape[-2:], mode="bilinear", align_corners=False)
            p3_gate = torch.sigmoid(self.p3_query(p3) * p4_up / math.sqrt(p3.shape[1]))
            output_p3 = self.p3_norm(p3 + p3_gate * p4_up)
            p3_down = F.adaptive_avg_pool2d(self.p3_to_p4(p3), p4.shape[-2:])
            p4_gate = torch.sigmoid(self.p4_query(p4) * p3_down / math.sqrt(p4.shape[1]))
            output_p4 = self.p4_norm(p4 + p4_gate * p3_down)
            return output_p3, output_p4


    class FM(nn.Module):
        """Focal modulation with local/global context for partially occluded apples."""

        def __init__(self, channels: int, levels: int = 3) -> None:
            super().__init__()
            self.query = nn.Conv2d(channels, channels, 1)
            self.context = nn.ModuleList([nn.Conv2d(channels, channels, 3 + 2 * level, padding=1 + level, groups=channels) for level in range(levels)])
            self.gates = nn.Conv2d(channels, levels + 1, 1)
            self.modulator = nn.Conv2d(channels, channels, 1)
            self.output = nn.Conv2d(channels, channels, 1)

        def forward(self, features: Any) -> Any:
            query = self.query(features)
            gates = torch.softmax(self.gates(features), dim=1)
            context: Any = 0
            current = features
            for level, layer in enumerate(self.context):
                current = layer(current)
                context = context + gates[:, level : level + 1] * current
            context = context + gates[:, -1:] * F.adaptive_avg_pool2d(features, 1)
            return self.output(query * self.modulator(context))


    class KWConv(nn.Module):
        """Kernel warehouse mixing 3×3, 5×5, and 7×7 kernels per input."""

        def __init__(self, channels: int, kernels: Sequence[int] = (3, 5, 7)) -> None:
            super().__init__()
            self.branches = nn.ModuleList([nn.Conv2d(channels, channels, kernel, padding=kernel // 2, bias=False) for kernel in kernels])
            self.selector = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, len(kernels), 1))
            self.norm = nn.BatchNorm2d(channels)
            self.activation = nn.SiLU()

        def forward(self, features: Any) -> Any:
            weights = torch.softmax(self.selector(features), dim=1)
            mixed = sum(weights[:, index : index + 1] * branch(features) for index, branch in enumerate(self.branches))
            return self.activation(self.norm(mixed))


    class ProposedEnhancementNeck(nn.Module):
        """AAM → EBCA → FM → KWConv on YOLO P3/P4/P5 features."""

        def __init__(self, p3_channels: int, p4_channels: int, p5_channels: int) -> None:
            super().__init__()
            self.aam = AAM(p4_channels)
            self.ebca = EBCA(p3_channels, p4_channels)
            self.fm = FM(p4_channels)
            self.kw_p3, self.kw_p4, self.kw_p5 = KWConv(p3_channels), KWConv(p4_channels), KWConv(p5_channels)

        def forward(self, p3: Any, p4: Any, p5: Any) -> tuple[Any, Any, Any]:
            p4 = self.aam(p4)
            p3, p4 = self.ebca(p3, p4)
            p4 = self.fm(p4)
            return self.kw_p3(p3), self.kw_p4(p4), self.kw_p5(p5)


    class EnhancedDetectHead(nn.Module):
        """Wrap Ultralytics Detect so it consumes the proposed enhanced P3/P4/P5."""

        def __init__(self, detect: Any, channels: Sequence[int]) -> None:
            super().__init__()
            self.detect = detect
            self.enhancement = ProposedEnhancementNeck(*channels)
            self.i, self.f = getattr(detect, "i", None), getattr(detect, "f", None)
            self.type = f"{self.__class__.__module__}.{self.__class__.__name__}"
            self.np = sum(parameter.numel() for parameter in self.parameters())

        @property
        def stride(self) -> Any:
            return self.detect.stride

        @property
        def nc(self) -> int:
            return self.detect.nc

        @property
        def nl(self) -> int:
            return self.detect.nl

        @property
        def no(self) -> int:
            return self.detect.no

        @property
        def reg_max(self) -> int:
            return self.detect.reg_max

        @property
        def dfl(self) -> Any:
            return self.detect.dfl

        @property
        def max_det(self) -> int:
            return self.detect.max_det

        @max_det.setter
        def max_det(self, value: int) -> None:
            self.detect.max_det = value

        def forward(self, features: Sequence[Any]) -> Any:
            return self.detect(list(self.enhancement(*features)))


else:

    class _TorchMissing:
        def __init__(self, *_: Any, **__: Any) -> None:
            _require_torch()

    class AAM(_TorchMissing):
        pass

    class EBCA(_TorchMissing):
        pass

    class FM(_TorchMissing):
        pass

    class KWConv(_TorchMissing):
        pass

    class ProposedEnhancementNeck(_TorchMissing):
        pass

    class EnhancedDetectHead(_TorchMissing):
        pass


def module_smoke_test(paths: ProjectPaths, device: str = "cpu") -> dict[str, Any]:
    """Verify P3/P4/P5 tensor interfaces and save an architecture report."""
    if torch is None:
        _require_torch()
    model = ProposedEnhancementNeck(64, 128, 256).to(device).eval()
    with torch.inference_mode():
        outputs = model(torch.zeros(1, 64, 80, 80, device=device), torch.zeros(1, 128, 40, 40, device=device), torch.zeros(1, 256, 20, 20, device=device))
    report = {"device": device, "output_shapes": [list(output.shape) for output in outputs], "module_order": ["AAM", "EBCA", "FM", "KWConv"]}
    _write_json(paths.architecture_results / "module_smoke_test.json", report)
    return report


def patch_yolov8_detection_head(yolo: Any) -> Any:
    """Insert the proposed feature neck immediately before a standard YOLOv8 Detect head."""
    if torch is None:
        _require_torch()
    candidate = getattr(yolo, "model", None)
    detection_model = candidate if candidate is not None and hasattr(candidate, "model") else yolo
    modules = getattr(detection_model, "model", None)
    if modules is None or not len(modules):
        raise TypeError("Expected an Ultralytics DetectionModel or YOLO object.")
    head = modules[-1]
    if isinstance(head, EnhancedDetectHead):
        return yolo
    try:
        channels = tuple(int(branch[0].conv.in_channels) for branch in head.cv2)
    except (AttributeError, TypeError, IndexError) as error:
        raise TypeError("Expected a standard three-scale Ultralytics Detect head.") from error
    if len(channels) != 3:
        raise ValueError(f"Expected P3/P4/P5 channels; got {channels}.")
    modules[-1] = EnhancedDetectHead(head, channels)
    return yolo


## AdaBins depth and depth-aware ByteTrack definitions

This cell contains the same AdaBins adapter, robust per-box relative-depth estimate, two-stage ByteTrack-style association, and duplicate-safe count used by the main notebook. AdaBins depth is **relative** unless an orchard depth calibration is supplied.

In [ ]:
# %% 02G) AdaBins relative depth, depth-aware ByteTrack, and unique count
class AdaBinsDepth:
    """Official AdaBins adapter; output is relative depth without orchard calibration."""

    def __init__(self, repository: str | Path, dataset: str = "kitti", device: str | None = None) -> None:
        repository_path = Path(repository).expanduser().resolve()
        if not (repository_path / "infer.py").exists():
            raise FileNotFoundError(f"AdaBins repository not found: {repository_path}")
        if str(repository_path) not in sys.path:
            sys.path.insert(0, str(repository_path))
        try:
            from infer import InferenceHelper
        except ModuleNotFoundError as error:
            raise RuntimeError("AdaBins dependencies are unavailable in this notebook kernel.") from error
        if torch is None:
            _require_torch()
        selected_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        previous = Path.cwd()
        try:
            os.chdir(repository_path)
            self.helper = InferenceHelper(dataset=dataset, device=selected_device)
        finally:
            os.chdir(previous)

    def predict(self, bgr_frame: np.ndarray) -> np.ndarray:
        rgb = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
        tensor = torch.from_numpy(rgb.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0)
        _, depth = self.helper.predict(tensor.to(self.helper.device))
        depth_map = np.asarray(depth).squeeze().astype(np.float32)
        return cv2.resize(depth_map, (bgr_frame.shape[1], bgr_frame.shape[0]), interpolation=cv2.INTER_LINEAR)


def robust_instance_depth(depth_map: np.ndarray | None, bbox: Sequence[float], trim_fraction: float = 0.10) -> float | None:
    """Return a robust central depth statistic for one detection box."""
    if depth_map is None:
        return None
    height, width = depth_map.shape[:2]
    x1, y1, x2, y2 = bbox
    margin_x, margin_y = (x2 - x1) * 0.15, (y2 - y1) * 0.15
    left, right = max(0, int(x1 + margin_x)), min(width, int(x2 - margin_x))
    top, bottom = max(0, int(y1 + margin_y)), min(height, int(y2 - margin_y))
    values = depth_map[top:bottom, left:right].reshape(-1) if right > left and bottom > top else np.array([])
    values = values[np.isfinite(values) & (values > 0)]
    if not len(values):
        return None
    values.sort()
    trim = int(len(values) * trim_fraction)
    if trim and len(values) > 2 * trim:
        values = values[trim:-trim]
    return float(np.median(values))


@dataclass(frozen=True)
class FruitDetection:
    bbox: tuple[float, float, float, float]
    score: float
    depth: float | None = None


@dataclass
class FruitTrack:
    track_id: int
    bbox: np.ndarray
    score: float
    depth: float | None
    hits: int = 1
    age: int = 1
    time_since_update: int = 0
    velocity: np.ndarray = field(default_factory=lambda: np.zeros(2, dtype=np.float32))
    updated: bool = True

    @property
    def center(self) -> np.ndarray:
        return np.array([(self.bbox[0] + self.bbox[2]) / 2, (self.bbox[1] + self.bbox[3]) / 2], dtype=np.float32)

    def predict(self) -> None:
        self.bbox[[0, 2]] += self.velocity[0]
        self.bbox[[1, 3]] += self.velocity[1]
        self.age += 1
        self.time_since_update += 1
        self.updated = False

    def update(self, detection: FruitDetection) -> None:
        previous_center = self.center
        self.bbox = np.asarray(detection.bbox, dtype=np.float32)
        self.velocity = 0.65 * self.velocity + 0.35 * (self.center - previous_center)
        self.score = detection.score
        self.depth = detection.depth if detection.depth is not None else self.depth
        self.hits += 1
        self.time_since_update = 0
        self.updated = True


class DepthAwareByteTracker:
    """ByteTrack-style high/low confidence association with relative-depth gating."""

    def __init__(self, *, high_confidence: float = 0.50, low_confidence: float = 0.10, iou_threshold: float = 0.10, max_cost: float = 0.70, depth_gate: float = 0.40, track_buffer: int = 30, min_hits: int = 3) -> None:
        self.high_confidence, self.low_confidence = high_confidence, low_confidence
        self.iou_threshold, self.max_cost, self.depth_gate = iou_threshold, max_cost, depth_gate
        self.track_buffer, self.min_hits = track_buffer, min_hits
        self.tracks: list[FruitTrack] = []
        self.next_id = 1

    def _cost(self, track: FruitTrack, detection: FruitDetection) -> float:
        overlap = _iou(track.bbox, detection.bbox)
        if overlap < self.iou_threshold:
            return float("inf")
        depth_cost = 0.0
        if track.depth is not None and detection.depth is not None:
            difference = abs(track.depth - detection.depth) / max(abs(track.depth), abs(detection.depth), 1e-6)
            if difference > self.depth_gate:
                return float("inf")
            depth_cost = difference / self.depth_gate
        scale = max(1.0, math.hypot(track.bbox[2] - track.bbox[0], track.bbox[3] - track.bbox[1]))
        detection_center = np.array([(detection.bbox[0] + detection.bbox[2]) / 2, (detection.bbox[1] + detection.bbox[3]) / 2])
        motion_cost = min(1.0, float(np.linalg.norm(track.center - detection_center) / (3 * scale)))
        return 0.65 * (1 - overlap) + 0.20 * depth_cost + 0.15 * motion_cost

    def _associate(self, track_indices: list[int], detections: list[FruitDetection], max_cost: float) -> tuple[list[tuple[int, int]], list[int], list[int]]:
        if not track_indices or not detections:
            return [], track_indices.copy(), list(range(len(detections)))
        costs = np.full((len(track_indices), len(detections)), 1e6, dtype=np.float64)
        for row, track_index in enumerate(track_indices):
            for column, detection in enumerate(detections):
                costs[row, column] = self._cost(self.tracks[track_index], detection)
        try:
            from scipy.optimize import linear_sum_assignment
            rows, columns = linear_sum_assignment(costs)
        except ModuleNotFoundError:
            rows, columns = np.array([], dtype=int), np.array([], dtype=int)
        matches, used_tracks, used_detections = [], set(), set()
        for row, column in zip(rows, columns):
            if costs[row, column] <= max_cost:
                matches.append((track_indices[int(row)], int(column)))
                used_tracks.add(track_indices[int(row)])
                used_detections.add(int(column))
        return matches, [item for item in track_indices if item not in used_tracks], [item for item in range(len(detections)) if item not in used_detections]

    def update(self, detections: Iterable[FruitDetection]) -> list[FruitTrack]:
        active = [item for item in detections if item.score >= self.low_confidence]
        for track in self.tracks:
            track.predict()
        high = [item for item in active if item.score >= self.high_confidence]
        low = [item for item in active if item.score < self.high_confidence]
        matches, unmatched_tracks, unmatched_high = self._associate(list(range(len(self.tracks))), high, self.max_cost)
        for track_index, detection_index in matches:
            self.tracks[track_index].update(high[detection_index])
        low_matches, _, _ = self._associate(unmatched_tracks, low, min(0.85, self.max_cost + 0.10))
        for track_index, detection_index in low_matches:
            self.tracks[track_index].update(low[detection_index])
        for detection_index in unmatched_high:
            detection = high[detection_index]
            self.tracks.append(FruitTrack(self.next_id, np.asarray(detection.bbox, dtype=np.float32), detection.score, detection.depth))
            self.next_id += 1
        self.tracks = [track for track in self.tracks if track.time_since_update <= self.track_buffer]
        return [track for track in self.tracks if track.updated and track.hits >= self.min_hits]


class UniqueLineCounter:
    """Count each confirmed track once when crossing a virtual line."""

    def __init__(self, start: tuple[int, int], end: tuple[int, int], direction: str = "both") -> None:
        if direction not in {"positive", "negative", "both"}:
            raise ValueError("direction must be positive, negative, or both")
        self.start, self.end, self.direction = np.array(start), np.array(end), direction
        self.counted_ids: set[int] = set()
        self.previous_sides: dict[int, float] = {}

    def update(self, tracks: Iterable[FruitTrack]) -> list[int]:
        newly_counted = []
        for track in tracks:
            point = track.center
            side = float(np.cross(self.end - self.start, point - self.start))
            previous = self.previous_sides.get(track.track_id)
            self.previous_sides[track.track_id] = side
            if previous is None or track.track_id in self.counted_ids or previous * side >= 0:
                continue
            valid = self.direction == "both" or (self.direction == "positive" and side > 0) or (self.direction == "negative" and side < 0)
            if valid:
                self.counted_ids.add(track.track_id)
                newly_counted.append(track.track_id)
        return newly_counted


def run_depth_aware_tracking(
    paths: ProjectPaths,
    *,
    weights: str | Path,
    adabins: AdaBinsDepth | None = None,
    confidence: float = 0.10,
    image_size: int = 1024,
    device: str | int = 0,
    line: tuple[tuple[int, int], tuple[int, int]] | None = None,
) -> dict[str, Any]:
    """Run enhanced YOLO, optional AdaBins depth, ByteTrack, and unique counting."""
    info = require_playable_video(paths.raw_video)
    YOLO = _require_ultralytics()
    model = YOLO(str(weights))
    capture = cv2.VideoCapture(str(paths.raw_video))
    output = paths.inference_results / "tracked_green_apples.mp4"
    writer = cv2.VideoWriter(str(output), cv2.VideoWriter_fourcc(*"mp4v"), float(info["source_fps"]), (int(info["width"]), int(info["height"])))
    if not writer.isOpened():
        raise OSError(f"Could not create {output}")
    tracker = DepthAwareByteTracker(low_confidence=confidence)
    counter = UniqueLineCounter(*line, direction="both") if line else None
    csv_path = paths.inference_results / "tracks.csv"
    frame_index = 0
    try:
        with csv_path.open("w", encoding="utf-8", newline="") as handle:
            fields = ("frame", "track_id", "score", "x1", "y1", "x2", "y2", "relative_depth", "unique_count")
            csv_writer = csv.DictWriter(handle, fieldnames=fields)
            csv_writer.writeheader()
            while True:
                ok, frame = capture.read()
                if not ok:
                    break
                prediction = model.predict(frame, conf=confidence, imgsz=image_size, device=device, verbose=False)[0]
                depth_map = adabins.predict(frame) if adabins is not None else None
                detections: list[FruitDetection] = []
                if prediction.boxes is not None:
                    for box in prediction.boxes:
                        if int(box.cls[0].item()) != GREEN_APPLE:
                            continue
                        bbox = tuple(float(value) for value in box.xyxy[0].tolist())
                        detections.append(FruitDetection(bbox, float(box.conf[0].item()), robust_instance_depth(depth_map, bbox)))
                tracks = tracker.update(detections)
                if counter:
                    counter.update(tracks)
                annotated = frame.copy()
                for track in tracks:
                    x1, y1, x2, y2 = (int(value) for value in track.bbox)
                    text = f"green_apple ID:{track.track_id}" + (f" d:{track.depth:.2f}" if track.depth is not None else "")
                    cv2.rectangle(annotated, (x1, y1), (x2, y2), (40, 210, 40), 2)
                    cv2.putText(annotated, text, (x1, max(20, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (40, 210, 40), 2)
                    csv_writer.writerow({"frame": frame_index, "track_id": track.track_id, "score": round(track.score, 6), "x1": x1, "y1": y1, "x2": x2, "y2": y2, "relative_depth": "" if track.depth is None else round(track.depth, 6), "unique_count": len(counter.counted_ids) if counter else ""})
                if counter:
                    cv2.line(annotated, tuple(counter.start), tuple(counter.end), (255, 255, 255), 2)
                    cv2.putText(annotated, f"Unique apples: {len(counter.counted_ids)}", (20, 36), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3)
                writer.write(annotated)
                frame_index += 1
    finally:
        capture.release()
        writer.release()
    summary = {"input_video": str(paths.raw_video), "frames_processed": frame_index, "weights": str(weights), "relative_depth_enabled": adabins is not None, "unique_count": len(counter.counted_ids) if counter else None, "output_video": str(output), "tracks_csv": str(csv_path)}
    _write_json(paths.inference_results / "tracking_summary.json", summary)
    return summary

# Existing short cells below use fp.function_name, but fp is this notebook module.
import sys
fp = sys.modules[__name__]
paths = fp.project_paths(PROJECT_ROOT)
paths.ensure_layout()
print('Notebook-only implementation loaded; no external project .py file is required.')


## Final validation: required plots and metrics CSV

Run only after all training epochs are complete. Ultralytics saves the F1-confidence, precision-confidence, recall-confidence, precision-recall, confusion matrix, and normalized confusion matrix. This cell copies each graph to one clear `Results/graphs/` folder and exports final scalar metrics to CSV.

In [ ]:
# STEP R1 — enable only after training has completed.
RUN_FINAL_VALIDATION = False
VALIDATION_RUN_NAME = 'final_enhanced_green_apple_validation'
BEST_WEIGHTS = PROJECT_ROOT / 'Results' / 'training' / RUN_NAME / 'weights' / 'best.pt'
GRAPHS_DIR = PROJECT_ROOT / 'Results' / 'graphs'
CSV_DIR = PROJECT_ROOT / 'Results' / 'csv'
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

if RUN_FINAL_VALIDATION:
    if not BEST_WEIGHTS.exists():
        raise FileNotFoundError(f'Trained enhanced checkpoint is missing: {BEST_WEIGHTS}')
    from ultralytics import YOLO
    import shutil
    import torch

    export_device = 0 if torch.cuda.is_available() else 'cpu'
    model = YOLO(str(BEST_WEIGHTS), task='detect')
    metrics = model.val(
        data=str(PROJECT_ROOT / 'Dataset' / 'yolo_green_apple' / 'green_apple.yaml'),
        imgsz=640, device=export_device, plots=True,
        project=str(PROJECT_ROOT / 'Results' / 'validation'), name=VALIDATION_RUN_NAME,
        exist_ok=True,
    )
    validation_dir = PROJECT_ROOT / 'Results' / 'validation' / VALIDATION_RUN_NAME
    required_plots = (
        'F1_curve.png', 'P_curve.png', 'R_curve.png', 'PR_curve.png',
        'confusion_matrix.png', 'confusion_matrix_normalized.png',
    )
    copied = []
    for filename in required_plots:
        source = validation_dir / filename
        if not source.exists():
            raise FileNotFoundError(f'Validation did not create the required graph: {source}')
        destination = GRAPHS_DIR / filename
        shutil.copy2(source, destination)
        copied.append(str(destination))
    metrics_dict = {str(key): value for key, value in metrics.results_dict.items()}
    metrics_csv = CSV_DIR / 'final_validation_metrics.csv'
    with metrics_csv.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=tuple(metrics_dict))
        writer.writeheader()
        writer.writerow(metrics_dict)
    shutil.copy2(RESULTS_CSV, CSV_DIR / 'training_epoch_metrics.csv')
    print(json.dumps({'validation_directory': str(validation_dir), 'graphs': copied, 'metrics_csv': str(metrics_csv)}, indent=2))
else:
    print('STEP R1 WAITING — training is active. Set RUN_FINAL_VALIDATION=True only after its final epoch and checkpoint are complete.')

## Three annotated videos, bounding-box CSV, confidence, AdaBins depth, and ByteTrack IDs

This is the final inference step. It processes the original `Dataset/DJI_0418.MP4` once at its source frame rate and creates exactly the three requested videos:

- `apples_id_depth.mp4`: bounding box + persistent ByteTrack ID + relative depth.
- `apples_confidence_depth.mp4`: bounding box + detector confidence + relative depth.
- `apples_id_confidence.mp4`: bounding box + persistent ByteTrack ID + detector confidence.

It also writes one row per raw detector box and one row per ByteTrack result. The boxes use original-video pixel coordinates. This uses the AdaBins checkpoint in `third_party/AdaBins/pretrained/AdaBins_kitti.pt`; its depth values are relative, not calibrated metres.

In [ ]:
# STEP R2 — enable only after training and final validation have completed.
RUN_VIDEO_EXPORTS = False
USE_ADABINS_RELATIVE_DEPTH = True
DETECTION_CONFIDENCE = 0.10
VIDEO_IMAGE_SIZE = 640
COUNT_LINE = ((0, 540), (1919, 540))

def _put_label(image, text, x, y, colour):
    import cv2
    x, y = max(0, int(x)), max(20, int(y))
    cv2.putText(image, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.52, colour, 2, cv2.LINE_AA)

def _draw_box(image, box, label, colour=(45, 220, 45)):
    import cv2
    x1, y1, x2, y2 = (int(value) for value in box)
    cv2.rectangle(image, (x1, y1), (x2, y2), colour, 2)
    _put_label(image, label, x1, y1 - 7, colour)

def export_requested_apple_videos(weights, *, use_adabins=True, confidence=0.10, image_size=640, line=None):
    import cv2
    import csv
    import json
    import torch
    from ultralytics import YOLO

    raw_video = PROJECT_ROOT / 'Dataset' / 'DJI_0418.MP4'
    if not raw_video.exists():
        raise FileNotFoundError(raw_video)
    if not Path(weights).exists():
        raise FileNotFoundError(weights)
    inference_dir = PROJECT_ROOT / 'Results' / 'inference'
    csv_dir = PROJECT_ROOT / 'Results' / 'csv'
    inference_dir.mkdir(parents=True, exist_ok=True)
    csv_dir.mkdir(parents=True, exist_ok=True)
    model = YOLO(str(weights), task='detect')
    device = 0 if torch.cuda.is_available() else 'cpu'
    depth_model = None
    if use_adabins:
        repository = PROJECT_ROOT / 'third_party' / 'AdaBins'
        checkpoint = repository / 'pretrained' / 'AdaBins_kitti.pt'
        if not checkpoint.exists():
            raise FileNotFoundError(f'AdaBins depth checkpoint is required but missing: {checkpoint}')
        depth_model = AdaBinsDepth(repository, dataset='kitti', device='cuda:0' if device != 'cpu' else 'cpu')
    capture = cv2.VideoCapture(str(raw_video))
    if not capture.isOpened():
        raise OSError(f'Cannot read {raw_video}')
    fps = float(capture.get(cv2.CAP_PROP_FPS))
    width, height = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    outputs = {
        'id_depth': inference_dir / 'apples_id_depth.mp4',
        'confidence_depth': inference_dir / 'apples_confidence_depth.mp4',
        'id_confidence': inference_dir / 'apples_id_confidence.mp4',
    }
    writers = {name: cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height)) for name, path in outputs.items()}
    if not all(writer.isOpened() for writer in writers.values()):
        capture.release()
        for writer in writers.values(): writer.release()
        raise OSError('Could not create all requested MP4 files in Results/inference/.')
    tracker = DepthAwareByteTracker(low_confidence=confidence)
    counter = UniqueLineCounter(*line, direction='both') if line else None
    detections_csv = csv_dir / 'apple_detections_confidence_depth.csv'
    tracks_csv = csv_dir / 'apple_tracks_id_confidence_depth.csv'
    frame_index = 0
    try:
        with detections_csv.open('w', newline='', encoding='utf-8') as det_handle, tracks_csv.open('w', newline='', encoding='utf-8') as track_handle:
            det_fields = ('frame', 'timestamp_seconds', 'detection_id', 'class_name', 'x1', 'y1', 'x2', 'y2', 'confidence', 'relative_depth')
            track_fields = ('frame', 'timestamp_seconds', 'track_id', 'x1', 'y1', 'x2', 'y2', 'confidence', 'relative_depth', 'track_hits', 'unique_count')
            det_writer, track_writer = csv.DictWriter(det_handle, fieldnames=det_fields), csv.DictWriter(track_handle, fieldnames=track_fields)
            det_writer.writeheader(); track_writer.writeheader()
            while True:
                ok, frame = capture.read()
                if not ok:
                    break
                prediction = model.predict(frame, conf=confidence, imgsz=image_size, device=device, verbose=False)[0]
                depth_map = depth_model.predict(frame) if depth_model is not None else None
                detections = []
                raw_rows = []
                if prediction.boxes is not None:
                    for detection_id, box in enumerate(prediction.boxes):
                        if int(box.cls[0].item()) != 0:
                            continue
                        bbox = tuple(float(value) for value in box.xyxy[0].tolist())
                        score = float(box.conf[0].item())
                        depth = robust_instance_depth(depth_map, bbox)
                        detections.append(FruitDetection(bbox, score, depth))
                        raw_rows.append((detection_id, bbox, score, depth))
                tracks = tracker.update(detections)
                if counter:
                    counter.update(tracks)
                id_depth_frame, confidence_depth_frame, id_confidence_frame = frame.copy(), frame.copy(), frame.copy()
                timestamp = frame_index / fps if fps else 0.0
                for detection_id, bbox, score, depth in raw_rows:
                    depth_text = 'NA' if depth is None else f'{depth:.3f}'
                    _draw_box(confidence_depth_frame, bbox, f'green_apple conf:{score:.2f} depth:{depth_text}', (0, 215, 255))
                    det_writer.writerow({'frame': frame_index, 'timestamp_seconds': round(timestamp, 6), 'detection_id': detection_id, 'class_name': 'green_apple', 'x1': round(bbox[0], 3), 'y1': round(bbox[1], 3), 'x2': round(bbox[2], 3), 'y2': round(bbox[3], 3), 'confidence': round(score, 6), 'relative_depth': '' if depth is None else round(depth, 6)})
                for track in tracks:
                    bbox = tuple(float(value) for value in track.bbox)
                    depth_text = 'NA' if track.depth is None else f'{track.depth:.3f}'
                    _draw_box(id_depth_frame, bbox, f'green_apple ID:{track.track_id} depth:{depth_text}', (50, 230, 50))
                    _draw_box(id_confidence_frame, bbox, f'green_apple ID:{track.track_id} conf:{track.score:.2f}', (255, 180, 0))
                    track_writer.writerow({'frame': frame_index, 'timestamp_seconds': round(timestamp, 6), 'track_id': track.track_id, 'x1': round(bbox[0], 3), 'y1': round(bbox[1], 3), 'x2': round(bbox[2], 3), 'y2': round(bbox[3], 3), 'confidence': round(track.score, 6), 'relative_depth': '' if track.depth is None else round(track.depth, 6), 'track_hits': track.hits, 'unique_count': len(counter.counted_ids) if counter else ''})
                if counter:
                    for image in (id_depth_frame, confidence_depth_frame, id_confidence_frame):
                        cv2.line(image, tuple(counter.start), tuple(counter.end), (255, 255, 255), 2)
                        _put_label(image, f'Unique apples: {len(counter.counted_ids)}', 20, 36, (255, 255, 255))
                writers['id_depth'].write(id_depth_frame)
                writers['confidence_depth'].write(confidence_depth_frame)
                writers['id_confidence'].write(id_confidence_frame)
                frame_index += 1
                if frame_index % 50 == 0 or frame_index == total_frames:
                    print(f'Processed {frame_index:,}/{total_frames:,} frames ({100 * frame_index / max(1, total_frames):.1f}%).')
    finally:
        capture.release()
        for writer in writers.values():
            writer.release()
    summary = {'source_video': str(raw_video), 'frames_processed': frame_index, 'source_fps': fps, 'width': width, 'height': height, 'weights': str(weights), 'adabins_relative_depth': depth_model is not None, 'unique_apple_count': len(counter.counted_ids) if counter else None, 'videos': {name: str(path) for name, path in outputs.items()}, 'detections_csv': str(detections_csv), 'tracks_csv': str(tracks_csv)}
    (inference_dir / 'apple_video_export_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    return summary

if RUN_VIDEO_EXPORTS:
    if not BEST_WEIGHTS.exists():
        raise FileNotFoundError(f'Trained enhanced checkpoint is missing: {BEST_WEIGHTS}')
    video_summary = export_requested_apple_videos(BEST_WEIGHTS, use_adabins=USE_ADABINS_RELATIVE_DEPTH, confidence=DETECTION_CONFIDENCE, image_size=VIDEO_IMAGE_SIZE, line=COUNT_LINE)
    print(json.dumps(video_summary, indent=2))
else:
    print('STEP R2 WAITING — set RUN_VIDEO_EXPORTS=True only after training and final validation have completed.')

## Final results inventory

Run after either result stage. It clearly lists which requested graph, video, and CSV file currently exists and which is still waiting for its corresponding step.

In [ ]:
expected = [
    PROJECT_ROOT / 'Results' / 'status' / 'live_training_status.json',
    PROJECT_ROOT / 'Results' / 'training' / RUN_NAME / 'results.csv',
    PROJECT_ROOT / 'Results' / 'csv' / 'training_epoch_metrics.csv',
    PROJECT_ROOT / 'Results' / 'csv' / 'final_validation_metrics.csv',
    PROJECT_ROOT / 'Results' / 'csv' / 'apple_detections_confidence_depth.csv',
    PROJECT_ROOT / 'Results' / 'csv' / 'apple_tracks_id_confidence_depth.csv',
    PROJECT_ROOT / 'Results' / 'graphs' / 'F1_curve.png',
    PROJECT_ROOT / 'Results' / 'graphs' / 'P_curve.png',
    PROJECT_ROOT / 'Results' / 'graphs' / 'R_curve.png',
    PROJECT_ROOT / 'Results' / 'graphs' / 'PR_curve.png',
    PROJECT_ROOT / 'Results' / 'graphs' / 'confusion_matrix.png',
    PROJECT_ROOT / 'Results' / 'graphs' / 'confusion_matrix_normalized.png',
    PROJECT_ROOT / 'Results' / 'inference' / 'apples_id_depth.mp4',
    PROJECT_ROOT / 'Results' / 'inference' / 'apples_confidence_depth.mp4',
    PROJECT_ROOT / 'Results' / 'inference' / 'apples_id_confidence.mp4',
    PROJECT_ROOT / 'Results' / 'inference' / 'apple_video_export_summary.json',
]
for artifact in expected:
    state = 'READY' if artifact.exists() else 'WAITING'
    size = f'{artifact.stat().st_size / 1024**2:.2f} MiB' if artifact.exists() else '-'
    print(f'{state:7} {size:>12}  {artifact.relative_to(PROJECT_ROOT)}')